In [1]:
%load_ext watermark
%watermark -a "Manuel Elias Orellana Lavayen" -d -v -iv

Author: Manuel Elias Orellana Lavayen

Date: 2026-10-05

Python implementation: CPython
Python version       : 3.12.10
IPython version      : 9.17.1

debugpy: 1.8.22



# Concatenando Dataset


In [2]:
# Librerias
import pandas as pd
import polars as pl
import os
from pathlib import Path
ruta_base = Path.cwd()
ruta_base

WindowsPath('C:/Users/ManuelOrellana/Desktop/Proyectos Aplicaciones/Aplicacion Prediccion Productos/Creacion de Modelos/Analisis y entrenamiento de modelos')

## Train

#### Train y Stores

In [4]:
diccionario_tipos_columnas = {
    "store_nbr": pl.Int8,
    "item_nbr": pl.Int32,
    "unit_sales": pl.Float32
}

# Dataset
df_train_stores = (
    pl.scan_csv(ruta_base / "Datasets/Modificados/train_rellenado.csv", schema_overrides={"onpromotion": pl.Boolean}, null_values=["NaN", "None", "null", ""] )
    .cast(diccionario_tipos_columnas)
    .drop("id")
    .join(pl.scan_csv(ruta_base / "Datasets/Originales/stores.csv").cast({"store_nbr": pl.Int8, "cluster": pl.Int8, "city": pl.Categorical,"state": pl.Categorical,"type": pl.Categorical}),
          on = "store_nbr",
          how= "left",
          maintain_order="left"))

# Guardado
df_train_stores.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/1_train_stores.parquet", engine="streaming")

print(f"{df_train_stores.select(pl.len()).collect().item()}")
df_train_stores.head(3).collect()

342784600


date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster
str,i8,i32,f32,bool,cat,cat,cat,i8
"""2013-01-01T00:00:00.000000""",25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1
"""2013-01-01T00:00:00.000000""",25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1
"""2013-01-01T00:00:00.000000""",25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1


#### Train_stores y transaction

In [5]:
diccionario_tipos_columnas = {
    "store_nbr": pl.Int8,
    "item_nbr": pl.Int32,
    "unit_sales": pl.Float32,
    "cluster": pl.Int8,
    "city": pl.Categorical,
    "state": pl.Categorical,
    "type": pl.Categorical
}

# Dataset
df_train_stores_transactions = \
    (pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/1_train_stores.parquet")
    .cast(diccionario_tipos_columnas)
     .with_columns(pl.col("date").str.slice(0, 10).str.to_date("%Y-%m-%d"))
     .join(pl.scan_csv(ruta_base / "Datasets/Originales/transactions.csv").cast({"store_nbr": pl.Int8, "transactions": pl.Float32}).with_columns( pl.col("date").str.to_date("%Y-%m-%d")),
           on=["date", "store_nbr"],
           how="left",
           maintain_order="left"))

# Guardado
df_train_stores_transactions.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/2_train_stores_transactions.parquet", engine="streaming")

print(f"{df_train_stores_transactions.select(pl.len()).collect().item()}")
df_train_stores_transactions.head(3).collect()

342784600


date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,transactions
date,i8,i32,f32,bool,cat,cat,cat,i8,f32
2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0
2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0
2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0


In [6]:
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/1_train_stores.parquet")

#### train_stores_transactions_items

In [7]:
diccionario_tipos_columnas = {
    "store_nbr": pl.Int8,
    "item_nbr": pl.Int32,
    "unit_sales": pl.Float32,
    "cluster": pl.Int8,
    "city": pl.Categorical,
    "state": pl.Categorical,
    "type": pl.Categorical,
    "transactions": pl.Float32
}

# Dataset
df_train_stores_transactions_items = \
    (pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/2_train_stores_transactions.parquet").cast(diccionario_tipos_columnas)
     .join(pl.scan_csv(ruta_base / "Datasets/Originales/items.csv").cast({"item_nbr":pl.Int32, "family": pl.Categorical ,"class":pl.Int16,"perishable": pl.Int8}),
          on = "item_nbr",
          how = "left",
          maintain_order="left"))

# Guardado
df_train_stores_transactions_items.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/3_train_stores_transactions_items.parquet", engine="streaming")

print(f"{df_train_stores_transactions_items.select(pl.len()).collect().item()}")
df_train_stores_transactions_items.head(3).collect()

342784600


date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,transactions,family,class,perishable
date,i8,i32,f32,bool,cat,cat,cat,i8,f32,cat,i16,i8
2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""BREAD/BAKERY""",2712,1
2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0
2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0


In [8]:
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/2_train_stores_transactions.parquet")

#### train_stores_transactions_items_Oil

In [9]:
diccionario_tipos_columnas = {
    "store_nbr": pl.Int8,
    "item_nbr": pl.Int32,
    "unit_sales": pl.Float32,
    "cluster": pl.Int8,
    "city": pl.Categorical,
    "state": pl.Categorical,
    "type": pl.Categorical,
    "transactions": pl.Float32,
    "family": pl.Categorical,
    "class":pl.Int16,
    "perishable": pl.Int8
}

# Dataset
df_train_stores_transactions_items_oil = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/3_train_stores_transactions_items.parquet").cast(diccionario_tipos_columnas)
    .join(pl.scan_csv(ruta_base / "Datasets/Modificados/oil_limpio.csv").cast({"dcoilwtico": pl.Float32}).with_columns( pl.col("date").str.to_date("%Y-%m-%d")),
        on = "date",
        how =  "left",
        maintain_order="left"))

# Guardado
df_train_stores_transactions_items_oil.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/4_train_stores_transactions_items_oil.parquet", engine="streaming")

print(f"{df_train_stores_transactions_items_oil.select(pl.len()).collect().item()}")
df_train_stores_transactions_items_oil.head(3).collect()

342784600


date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,transactions,family,class,perishable,dcoilwtico
date,i8,i32,f32,bool,cat,cat,cat,i8,f32,cat,i16,i8,f32
2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""BREAD/BAKERY""",2712,1,94.279999
2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999
2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999


In [10]:
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/3_train_stores_transactions_items.parquet")

#### train_stores_transactions_items_oil_holidays

In [11]:
diccionario_tipos_columnas = {
    "store_nbr": pl.Int8,
    "item_nbr": pl.Int32,
    "unit_sales": pl.Float32,
    "cluster": pl.Int8,
    "city": pl.Categorical,
    "state": pl.Categorical,
    "type": pl.Categorical,
    "transactions": pl.Float32,
    "family": pl.Categorical,
    "class":pl.Int16,
    "perishable": pl.Int8,
    "dcoilwtico": pl.Float32
}

###### Nacional

In [12]:
# Dataset
df_train_stores_transactions_items_oil_holidays_nacional = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/4_train_stores_transactions_items_oil.parquet").cast(diccionario_tipos_columnas)
    .join(pl.scan_csv(ruta_base / "Datasets/Modificados/holidays_nacional.csv").with_columns( pl.col("date").str.to_date("%Y-%m-%d")),
        on = ["date"],
        how = "left",
        maintain_order="left")
    .with_columns([pl.col("es_feriado_nacional").fill_null(False), pl.col("es_evento").fill_null(False)]))

# Guardado
df_train_stores_transactions_items_oil_holidays_nacional.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/5.1_train_stores_transactions_items_oil_holidays_nacional.parquet", engine="streaming")

print(f"{df_train_stores_transactions_items_oil_holidays_nacional.select(pl.len()).collect().item()}")
df_train_stores_transactions_items_oil_holidays_nacional.head(3).collect()

342784600


date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,transactions,family,class,perishable,dcoilwtico,es_feriado_nacional,es_evento
date,i8,i32,f32,bool,cat,cat,cat,i8,f32,cat,i16,i8,f32,bool,bool
2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""BREAD/BAKERY""",2712,1,94.279999,true,false
2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false
2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false


In [13]:
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/4_train_stores_transactions_items_oil.parquet")

###### Regional

In [14]:
# Dataset
df_train_stores_transactions_items_oil_holidays_nacional_regional = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/5.1_train_stores_transactions_items_oil_holidays_nacional.parquet").cast(diccionario_tipos_columnas)
    .join(pl.scan_csv(ruta_base / "Datasets/Modificados/holidays_regional.csv").with_columns( pl.col("date").str.to_date("%Y-%m-%d")).cast({"locale_name": pl.Categorical}).rename({"locale_name": "state"}),
        on = ["date", "state"],
        how = "left",
        maintain_order="left")
    .with_columns(pl.col("es_feriado_regional").fill_null(False)))

# Guardado
df_train_stores_transactions_items_oil_holidays_nacional_regional.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/5.2_train_stores_transactions_items_oil_holidays_nacional_regional.parquet", engine="streaming")

print(f"{df_train_stores_transactions_items_oil_holidays_nacional_regional.select(pl.len()).collect().item()}")
df_train_stores_transactions_items_oil_holidays_nacional_regional.head(3).collect()

342784600


date,store_nbr,item_nbr,unit_sales,onpromotion,city,state,type,cluster,transactions,family,class,perishable,dcoilwtico,es_feriado_nacional,es_evento,es_feriado_regional
date,i8,i32,f32,bool,cat,cat,cat,i8,f32,cat,i16,i8,f32,bool,bool,bool
2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""BREAD/BAKERY""",2712,1,94.279999,true,false,false
2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false,false
2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false,false


In [15]:
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/5.1_train_stores_transactions_items_oil_holidays_nacional.parquet")

###### Local

In [16]:
diccionario_nombres_columnas = {
    'date': 'fecha',
    'store_nbr': 'numero_tienda',
    'item_nbr': 'numero_articulo',
    'unit_sales': 'unidades_vendidas',
    'onpromotion': 'en_promocion',
    'transactions': 'transacciones',
    'city': 'ciudad',
    'state': 'provincia_estado',
    'type': 'tipo_tienda',
    'cluster': 'cluster',
    'family': 'familia_producto',
    'class': 'clase_producto',
    'perishable': 'perecedero',
    'dcoilwtico': 'precio_petroleo_wti'  # Refiere al precio del crudo WTI (West Texas Intermediate)
}
# Dataset
df_train_stores_transactions_items_oil_holidays_nacional_regional_local = (
    pl.scan_parquet(ruta_base / "Datasets/Concatenados/Sucios/5.2_train_stores_transactions_items_oil_holidays_nacional_regional.parquet").cast(diccionario_tipos_columnas)
    .join(pl.scan_csv(ruta_base / "Datasets/Modificados/holidays_local.csv").with_columns( pl.col("date").str.to_date("%Y-%m-%d")).cast({"locale_name": pl.Categorical}).rename({"locale_name": "city"}),
        on = ["date", "city"],
        how = "left",
        maintain_order="left")
    .with_columns(pl.col("es_feriado_local").fill_null(False))
    .rename(diccionario_nombres_columnas)
    .with_row_index("ID"))

# Guardado
df_train_stores_transactions_items_oil_holidays_nacional_regional_local.sink_parquet(ruta_base / "Datasets/Concatenados/Sucios/train.parquet")

print(f"{df_train_stores_transactions_items_oil_holidays_nacional_regional_local.select(pl.len()).collect().item()}")
df_train_stores_transactions_items_oil_holidays_nacional_regional_local.head(3).collect()

342784600


ID,fecha,numero_tienda,numero_articulo,unidades_vendidas,en_promocion,ciudad,provincia_estado,tipo_tienda,cluster,transacciones,familia_producto,clase_producto,perecedero,precio_petroleo_wti,es_feriado_nacional,es_evento,es_feriado_regional,es_feriado_local
u32,date,i8,i32,f32,bool,cat,cat,cat,i8,f32,cat,i16,i8,f32,bool,bool,bool,bool
0,2013-01-01,25,103665,7.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""BREAD/BAKERY""",2712,1,94.279999,true,false,false,false
1,2013-01-01,25,105574,1.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false,false,false
2,2013-01-01,25,105575,2.0,null,"""Salinas""","""Santa Elena""","""D""",1,770.0,"""GROCERY I""",1045,0,94.279999,true,false,false,false


In [17]:
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/5.2_train_stores_transactions_items_oil_holidays_nacional_regional.parquet")

## Test

#### test_stores

In [18]:
# Dataset
df_test = pd.read_csv(filepath_or_buffer= ruta_base / "Datasets/Originales/test.csv", low_memory=False)
df_stores = pd.read_csv(filepath_or_buffer= ruta_base / "Datasets/Originales/stores.csv")

# Concatenación
df_test_stores = pd.merge(
    df_test, df_stores,
    on = "store_nbr",
    how = "left"
)

# Guardado
df_test_stores.to_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/1_test_stores.parquet")

print(df_test_stores.shape)
df_test_stores.head(3)

(3370464, 9)


,id,date,store_nbr,item_nbr,onpromotion,city,state,type,cluster
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13
1,125497041,2017-08-16,1,99197,False,Quito,Pichincha,D,13
2,125497042,2017-08-16,1,103501,False,Quito,Pichincha,D,13


#### test_stores_items

In [19]:
# Dataset
df_test_stores = pd.read_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/1_test_stores.parquet")
df_items = pd.read_csv(filepath_or_buffer= ruta_base / "Datasets/Originales/items.csv")

# Concatenación
df_test_stores_items = pd.merge(
    df_test_stores, df_items,
    on = "item_nbr",
    how = "left"
)

# Guardado
df_test_stores_items.to_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/2_test_stores_items.parquet")
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/1_test_stores.parquet")

print(df_test_stores_items.shape)
df_test_stores_items.head(3)

(3370464, 12)


,id,date,store_nbr,item_nbr,onpromotion,city,state,type,cluster,family,class,perishable
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0
1,125497041,2017-08-16,1,99197,False,Quito,Pichincha,D,13,GROCERY I,1067,0
2,125497042,2017-08-16,1,103501,False,Quito,Pichincha,D,13,CLEANING,3008,0


#### test_stores_items_Oil

In [20]:
# Dataset
df_test_stores_items = pd.read_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/2_test_stores_items.parquet")
df_oil = pd.read_csv(filepath_or_buffer= ruta_base / "Datasets/Modificados/oil_limpio.csv")

# Concatenación
df_test_stores_items_oil = pd.merge(
    df_test_stores_items, df_oil,
    on = "date",
    how =  "left"
)

# Guardado
df_test_stores_items_oil.to_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/3_test_stores_items_oil.parquet")
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/2_test_stores_items.parquet")

print(df_test_stores_items_oil.shape)
df_test_stores_items_oil.head(3)

(3370464, 13)


,id,date,store_nbr,item_nbr,onpromotion,city,state,type,cluster,family,class,perishable,dcoilwtico
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,46.8
1,125497041,2017-08-16,1,99197,False,Quito,Pichincha,D,13,GROCERY I,1067,0,46.8
2,125497042,2017-08-16,1,103501,False,Quito,Pichincha,D,13,CLEANING,3008,0,46.8


#### test_stores_items_Oil_holidays

###### Nacional

In [21]:
# Dataset
df_test_stores_items_oil = pd.read_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/3_test_stores_items_oil.parquet")
df_holidays_events_nacional = pd.read_csv(ruta_base / "Datasets/Modificados/holidays_nacional.csv")

df_test_stores_items_oil_holidays_nacional = pd.merge(
    df_test_stores_items_oil, df_holidays_events_nacional,
    on = "date",
    how = "left"
)
df_test_stores_items_oil_holidays_nacional["es_feriado_nacional"] = df_test_stores_items_oil_holidays_nacional["es_feriado_nacional"].fillna(False)
df_test_stores_items_oil_holidays_nacional["es_evento"] = df_test_stores_items_oil_holidays_nacional["es_evento"].fillna(False)

# Guardado
df_test_stores_items_oil_holidays_nacional.to_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/4.1_test_stores_items_oil_holidays_nacional.parquet")
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/3_test_stores_items_oil.parquet")

print(df_test_stores_items_oil_holidays_nacional.shape)
df_test_stores_items_oil_holidays_nacional.head(3)

(3370464, 15)


,id,date,store_nbr,item_nbr,onpromotion,city,state,type,cluster,family,class,perishable,dcoilwtico,es_feriado_nacional,es_evento
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,46.8,False,False
1,125497041,2017-08-16,1,99197,False,Quito,Pichincha,D,13,GROCERY I,1067,0,46.8,False,False
2,125497042,2017-08-16,1,103501,False,Quito,Pichincha,D,13,CLEANING,3008,0,46.8,False,False


###### Regional

In [22]:
# Dataset
df_test_stores_items_oil_holidays_nacional = pd.read_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/4.1_test_stores_items_oil_holidays_nacional.parquet")
df_holidays_eventos_regional = pd.read_csv(ruta_base / "Datasets/Modificados/holidays_regional.csv")
df_holidays_eventos_regional = df_holidays_eventos_regional.rename(columns= {"locale_name": "state"})

df_test_stores_items_oil_holidays_nacional_regional = pd.merge(
    df_test_stores_items_oil_holidays_nacional, df_holidays_eventos_regional,
    on = ["date", "state"],
    how = "left"
)
df_test_stores_items_oil_holidays_nacional_regional["es_feriado_regional"] = df_test_stores_items_oil_holidays_nacional_regional["es_feriado_regional"].fillna(False)

# Guardado
df_test_stores_items_oil_holidays_nacional_regional.to_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/4.2_test_stores_items_oil_holidays_nacional_regional.parquet")
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/4.1_test_stores_items_oil_holidays_nacional.parquet")

print(df_test_stores_items_oil_holidays_nacional_regional.shape)
df_test_stores_items_oil_holidays_nacional_regional.head(3)

(3370464, 16)


,id,date,store_nbr,item_nbr,onpromotion,city,state,type,cluster,family,class,perishable,dcoilwtico,es_feriado_nacional,es_evento,es_feriado_regional
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,46.8,False,False,False
1,125497041,2017-08-16,1,99197,False,Quito,Pichincha,D,13,GROCERY I,1067,0,46.8,False,False,False
2,125497042,2017-08-16,1,103501,False,Quito,Pichincha,D,13,CLEANING,3008,0,46.8,False,False,False


###### Local

In [23]:
# Dataset
df_test_stores_items_oil_holidays_nacional_regional = pd.read_parquet(path= ruta_base /  "Datasets/Concatenados/Sucios/4.2_test_stores_items_oil_holidays_nacional_regional.parquet")
df_holidays_eventos_local = pd.read_csv(ruta_base / "Datasets/Modificados/holidays_local.csv")
df_holidays_eventos_local = df_holidays_eventos_local.rename(columns= {"locale_name": "city"})

df_test_stores_items_oil_holidays_nacional_regional_local = pd.merge(
    df_test_stores_items_oil_holidays_nacional_regional, df_holidays_eventos_local,
    on = ["date", "city"],
    how = "left"
)

df_test_stores_items_oil_holidays_nacional_regional_local["es_feriado_local"] = df_test_stores_items_oil_holidays_nacional_regional_local["es_feriado_local"].fillna(False)

diccionario_nombres_columnas_2 = {
    'id': 'id',
    'date': 'fecha',
    'store_nbr': 'numero_tienda',
    'item_nbr': 'numero_articulo',
    'onpromotion': 'en_promocion',
    'city': 'ciudad',
    'state': 'provincia_estado',
    'type': 'tipo_tienda',
    'cluster': 'cluster',
    'family': 'familia_producto',
    'class': 'clase_producto',
    'perishable': 'perecedero',
    'dcoilwtico': 'precio_petroleo_wti'  # Refiere al precio del crudo WTI (West Texas Intermediate)
}

df_test_stores_items_oil_holidays_nacional_regional_local = df_test_stores_items_oil_holidays_nacional_regional_local.rename(columns= diccionario_nombres_columnas_2)

# Guardado
df_test_stores_items_oil_holidays_nacional_regional_local.to_parquet(path= ruta_base / "Datasets/Concatenados/Sucios/test.parquet")
# Eliminando Concatenado Anterior
os.remove(ruta_base / "Datasets/Concatenados/Sucios/4.2_test_stores_items_oil_holidays_nacional_regional.parquet")

print(df_test_stores_items_oil_holidays_nacional_regional_local.shape)
df_test_stores_items_oil_holidays_nacional_regional_local.head(3)

(3370464, 17)


,id,fecha,numero_tienda,numero_articulo,en_promocion,ciudad,provincia_estado,tipo_tienda,cluster,familia_producto,clase_producto,perecedero,precio_petroleo_wti,es_feriado_nacional,es_evento,es_feriado_regional,es_feriado_local
0,125497040,2017-08-16,1,96995,False,Quito,Pichincha,D,13,GROCERY I,1093,0,46.8,False,False,False,False
1,125497041,2017-08-16,1,99197,False,Quito,Pichincha,D,13,GROCERY I,1067,0,46.8,False,False,False,False
2,125497042,2017-08-16,1,103501,False,Quito,Pichincha,D,13,CLEANING,3008,0,46.8,False,False,False,False
